### NOTEBOOK DEMO

if you want to training just with 1 run, visit in kaggle (requrire Wandb API):
https://www.kaggle.com/code/khoangminh/dsp391m-group1-wap-running

NOTE: this project can only run in your own kaggle if you choose "Copy and edit" notebook, otherwise it maybe crash because this notebook have python ver = 3.10, but a new notebook python vers = 3.11

# ------------------ GET STARTED ------------------- #

<h5> Import wandb API key from kaggle secrets </h5>

In [ ]:
from kaggle_secrets import UserSecretsClient
from IPython.display import clear_output
user_secrets = UserSecretsClient()
wandb_token = user_secrets.get_secret("wandb_api")

<h5>Clone datasets and repo</h5>

In [ ]:
!git clone https://github.com/Purin1410/DSP391m_Group1_FALL25.git -b WAP
!git clone https://github.com/Purin1410/DSP391m_Group1_FALL25.git -d main
%cd /kaggle/working/DSP391m_Group1_FALL25
!unzip /kaggle/working/main/DSP391m_Group1_FALL25/crohme_data.zip
clear_output()

<h5>Install dependency</h5>

In [ ]:
!pip install -q sconf
!pip install -q pytorch==1.8.1 torchvision==0.2.2 cudatoolkit==11.1 pillow==8.4.0
!pip install -q pytorch-lightning==1.4.9 torchmetrics==0.6.0
!pip install -q pandoc==1.19.2.1
!pip install -q einops
!pip install -r requirements.txt
clear_output()

<h5>Overwrite the config if we want to change datasets or hyperpparameters</h5>

In [ ]:
%%writefile config/crohme_config.yaml
seed_everything: 7

trainer:
  accelerator: ddp #gpu           
  devices: 0,1            
  precision: 32
  deterministic: true
  max_epochs: 300
  check_val_every_n_epoch: 2
  val_check_interval: 1.0
  log_every_n_steps: 50
  num_sanity_val_steps: 2
  accumulate_grad_batches: 1
  resume_from_checkpoint: null

  early_stopping: false
  # early_stopping:
  #   monitor: val_loss
  #   mode: min
  #   patience: 15
  #   min_delta: 0.0

  checkpoint:
    dirpath: checkpoints
    filename: "{epoch}-{val_ExpRate:.4f}"
    monitor: val_ExpRate
    mode: max
    save_top_k: 1
    save_last: true

  log_grad_norm: true
  log_lr: true
  lr_logging_interval: epoch

model:
  # ----- Encoder -----
  input_channels: 1
  dim_ConvBlock: [32, 64, 64, 128]
  layersNum_block: [4, 4, 4, 4]
  kernel_Convenc: [3, 3]
  use_dropout: true
  encoder_dropout_p: 0.2
  block_dropout_indices: [2, 3]

  # ----- Decoder -----
  dim_target: 114        # vocab size
  dim_word: 256
  dim_dec: 256
  dim_attention: 128
  use_coverage: true
  dim_coverage: 128
  kernel_coverage: [5, 5]
  maxout_groups: 2

  # ----- Beam search -----
  beam_size: 5
  max_len: 200
  alpha: 0.0
  early_stopping: true
  temperature: 1.0

  # ----- Optimizer / Training -----
  learning_rate: 0.08
  weight_decay: 1e-4
  betas: [0.9, 0.999]
  use_focal_loss: false
  focal_alpha: 1.0
  focal_gamma: 2.0

  momentum: 0.9
  scheduler:
    factor: 0.25
    patience: 12
    mode: max


  patience: 15

data:
  zipfile_path: crohme_data/data
  test_year: '2014'
  dictionary_txt: crohme_data/data/dictionary.txt
  # caption_txt: crohme_data/data/crohme_caption.txt
  train_batch_size: 512
  eval_batch_size: 512
  num_workers: 4
  scale_aug: false

  # Augmentation params
  k_min: 0.7
  k_max: 1.4
  h_lo: 16
  h_hi: 256
  w_lo: 16
  w_hi: 1024

  # Memory control
  gpu_max_memory: 1280000  
  pin_memory: true
  persistent_workers: true
  free_memory: false

wandb:
  name: "WAP"
  project: "DSP391m_Group1_FALL25"

<h5>Logging and training</h5>

In [ ]:
import os
os.system(f"wandb login {wandb_token}")
!python train.py --config configs/crohme_config.yaml